# Victoria Map / Location Data Cleaning

## Purpose

This notebook cleans and validates the Victoria map/location dataset used by the FireFusion project.

The objective is to identify and fix:

- Missing values
- Empty fields
- Inconsistent formatting
- Duplicate records
- Duplicate location records
- Invalid latitude or longitude values
- Other location-related data quality issues

The cleaning process is performed programmatically so that all changes are reproducible and can be documented.

In [41]:
from pathlib import Path
import json
import pandas as pd

## 1. Load the Victoria Location Dataset

The source dataset contains location information including:

- Latitude
- Longitude
- Electorate rating
- Suburb
- Postcode
- State

The dataset is loaded into a pandas DataFrame for profiling and cleaning.

In [42]:
project_root = Path.cwd().parent

file_path = project_root / "datasets" / "victoria_suburbs_geo.json"

print("Dataset path:", file_path)
print("File exists:", file_path.exists())

with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

df = pd.DataFrame(data)

df.head()

Dataset path: /Users/architchandna/Desktop/data-engineering/datasets/victoria_suburbs_geo.json
File exists: True


,latitude,longitude,electoraterating,suburb,postcode,state
0,-37.815207,144.963937,Inner Metropolitan,MELBOURNE,3000,VIC
1,-37.813628,144.963058,,MELBOURNE,3001,VIC
2,-37.816144,144.980459,Inner Metropolitan,EAST MELBOURNE,3002,VIC
3,-37.811450,144.925397,Inner Metropolitan,WEST MELBOURNE,3003,VIC
4,-37.830158,144.980459,Inner Metropolitan,MELBOURNE,3004,VIC


## 2. Dataset Overview

Before cleaning the data, the structure of the dataset is inspected.

This includes checking:

- Number of records
- Number of columns
- Column names
- Data types

In [43]:
print(f"Total records: {len(df)}")
print(f"Total columns: {len(df.columns)}")

print("\nColumn names:")
for column in df.columns:
    print("-", column)

print("\nData types:")
print(df.dtypes)

Total records: 3540
Total columns: 6

Column names:
- latitude
- longitude
- electoraterating
- suburb
- postcode
- state

Data types:
latitude            float64
longitude           float64
electoraterating     object
suburb               object
postcode             object
state                object
dtype: object


## 3. Missing and Empty Values

The dataset is checked for both standard null values and empty strings.

This is important because some JSON datasets represent missing text values as an empty string (`""`) rather than a null value.

In [44]:
print("Null values:")
print(df.isnull().sum())

print("\nEmpty string values:")

for column in df.columns:
    if df[column].dtype == "object":
        empty_count = (
            df[column]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )

        print(f"{column}: {empty_count}")

Null values:
latitude            0
longitude           0
electoraterating    0
suburb              0
postcode            0
state               0
dtype: int64

Empty string values:
electoraterating: 83
suburb: 0
postcode: 0
state: 0


## 4. Duplicate Records

The dataset is checked for completely duplicated rows.

Exact duplicates can cause inaccurate location counts and may create unnecessary duplicate records during database integration.

In [45]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate records: {duplicate_rows}")

Duplicate records: 1


## 5. Coordinate Validation

Latitude and longitude values are validated to identify records with obviously invalid geographical coordinates.

General valid coordinate ranges are:

- Latitude: -90 to 90
- Longitude: -180 to 180

In [46]:
print("Latitude range:")
print(df["latitude"].min(), "to", df["latitude"].max())

print("\nLongitude range:")
print(df["longitude"].min(), "to", df["longitude"].max())

invalid_coordinates = df[
    (~df["latitude"].between(-90, 90)) |
    (~df["longitude"].between(-180, 180))
]

print(f"\nInvalid coordinate records: {len(invalid_coordinates)}")

invalid_coordinates

Latitude range:
-39.0317235 to 49.1805

Longitude range:
-122.2358302 to 152.663801

Invalid coordinate records: 0


,latitude,longitude,electoraterating,suburb,postcode,state


## 6. Duplicate Coordinates

Some locations may legitimately share the same coordinates, so duplicate coordinates should not be removed automatically.

This check identifies records sharing the same latitude and longitude so they can be reviewed before cleaning.

In [47]:
duplicate_coordinates = df[
    df.duplicated(
        subset=["latitude", "longitude"],
        keep=False
    )
].sort_values(
    by=["latitude", "longitude"]
)

print(
    f"Records sharing coordinates: "
    f"{len(duplicate_coordinates)}"
)

duplicate_coordinates.head(30)

Records sharing coordinates: 198


,latitude,longitude,electoraterating,suburb,postcode,state
3374,-38.737790,145.908715,Rural,TARWIN,3956,VIC
3375,-38.737790,145.908715,Rural,TARWIN LOWER,3956,VIC
3503,-38.608950,145.591103,Rural,NORTH WONTHAGGI,3995,VIC
3508,-38.608950,145.591103,Rural,WONTHAGGI,3995,VIC
3100,-38.395060,146.149841,Rural,MIRBOO EAST,3870,VIC
3108,-38.395060,146.149841,Rural,MIRBOO,3871,VIC
3109,-38.395060,146.149841,Rural,MIRBOO NORTH,3871,VIC
3373,-38.395060,146.149841,Rural,MIRBOO SOUTH,3956,VIC
3284,-38.394300,145.209000,Rural,FLINDERS NAVAL DEPOT,3920,VIC
3285,-38.394300,145.209000,Rural,HMAS CERBERUS,3920,VIC


## 7. Electorate Rating Validation

The initial inspection of the dataset showed that some `electoraterating` values may be empty.

This field is analysed to identify the number of missing values and the valid categories present in the dataset.

In [48]:
print("Electorate rating counts:")

print(
    df["electoraterating"]
    .replace("", pd.NA)
    .value_counts(dropna=False)
)

Electorate rating counts:
electoraterating
Rural                 2610
Provincial             335
Inner Metropolitan     270
Outer Metropolitan     242
<NA>                    83
Name: count, dtype: int64


## 8. Suburb Formatting Validation

Suburb names are checked for inconsistent formatting such as:

- Leading or trailing spaces
- Empty values
- Mixed capitalisation
- Repeated internal whitespace

Consistent suburb naming is important for matching location records with other FireFusion datasets.

In [49]:
print(
    "Empty suburb values:",
    df["suburb"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

whitespace_suburbs = df[
    df["suburb"] != df["suburb"].astype(str).str.strip()
]

print(
    "Suburbs with leading/trailing whitespace:",
    len(whitespace_suburbs)
)

print(
    "Unique suburb values:",
    df["suburb"].nunique()
)

Empty suburb values: 0
Suburbs with leading/trailing whitespace: 0
Unique suburb values: 3351


## 9. Data Cleaning

Based on the profiling results, the following cleaning operations are applied:

- Remove exact duplicate records.
- Replace empty `electoraterating` values with `"Unknown"`.
- Remove leading and trailing whitespace from text fields.
- Standardise postcode formatting by ensuring it is stored as a string.
- Preserve valid duplicate coordinates because multiple suburbs may legitimately share the same location.

The cleaned dataset is stored separately to preserve the original source data.

In [50]:
clean_df = df.copy()

# Remove duplicate rows
clean_df = clean_df.drop_duplicates()

# Replace empty electorate ratings
clean_df["electoraterating"] = (
    clean_df["electoraterating"]
    .replace("", "Unknown")
)

# Remove leading/trailing whitespace
text_columns = [
    "electoraterating",
    "suburb",
    "postcode",
    "state"
]

for column in text_columns:
    clean_df[column] = (
        clean_df[column]
        .astype(str)
        .str.strip()
    )

# Ensure postcode remains a string
clean_df["postcode"] = clean_df["postcode"].astype(str)

print("Cleaning complete.")
print(f"Total records after cleaning: {len(clean_df)}")

Cleaning complete.
Total records after cleaning: 3539


## 10. Validation After Cleaning

The cleaned dataset is validated to confirm that the identified issues have been resolved.

In [51]:
print("Duplicate rows:", clean_df.duplicated().sum())

print("\nEmpty electorate ratings:")
print(
    (clean_df["electoraterating"]
     .str.strip()
     .eq(""))
    .sum()
)

print("\nUnknown electorate ratings:")
print(
    (clean_df["electoraterating"] == "Unknown").sum()
)

Duplicate rows: 0

Empty electorate ratings:
0

Unknown electorate ratings:
83


## 11. Export Cleaned Dataset

The cleaned dataset is exported as a new JSON file.
The original dataset remains unchanged.

In [52]:
output_path = (
    project_root /
    "datasets" /
    "victoria_suburbs_geo_cleaned.json"
)

clean_df.to_json(
    output_path,
    orient="records",
    indent=4
)

print(f"Cleaned dataset saved to:\n{output_path}")

Cleaned dataset saved to:
/Users/architchandna/Desktop/data-engineering/datasets/victoria_suburbs_geo_cleaned.json


# 12. Cleaning Summary

The Victoria location dataset was successfully profiled and cleaned.

## Summary of Changes

- Removed **1** duplicate record.
- Replaced **83** empty `electoraterating` values with `"Unknown"`.
- Standardised text formatting by removing leading and trailing whitespace.
- Confirmed that all latitude and longitude values were valid.
- Preserved duplicate coordinates because they represent legitimate geographical relationships between suburbs.

The cleaned dataset is now suitable for database integration and downstream location-based processing within the FireFusion project.

# 13. Cleaning Report

| Issue | Records | Action Taken |
|-------|--------:|--------------|
| Missing values | 0 | No action required |
| Empty electorate ratings | 83 | Replaced with "Unknown" |
| Duplicate rows | 1 | Removed |
| Invalid coordinates | 0 | No action required |
| Duplicate coordinates | 198 | Reviewed and retained as valid |
| Leading/trailing whitespace | 0 | No action required |

## Outcome

The cleaned dataset is suitable for downstream processing, database integration and location matching within the FireFusion project.